#### Welcome to FOS Deployment

This notebook deployes the latest FOS version in the specified workspace. It works for initial deployment and for the upgrade process of FOS.

**End-to-end documenation on fabric-toolbox:**

[Visit - How to deploy and configure FOS](https://github.com/microsoft/fabric-toolbox/blob/main/monitoring/fabric-unified-admin-monitoring/how-to/How_to_deploy_FOS.md)

**What is happening in this notebook?**
 - The notebook checks the two cloud connections for FOS (if initial deployment, connections will be created, otherwise check only)
 - It downloads the latest FOS src files from Github
 - It deploys/updates the Fabric items in the current workspace
 - It creates all needed tables automatically, so reports work also with some data missing

**Next steps**
- (Optional) Change connection names, only if needed
- Run this notebook

If you **deploy** FOS in this workspace at the **first time**:
- Navigate to the cloud connections
- Search under cloud connection for **fos fabric-service-api admin** and for **fos pbi-service-api admin** 
- Add the credentials of your service principal to these connections

If you **update** your existing FOS workspace:
- After the notebooks has been executed, you are **done**


In [ ]:
%pip install ms-fabric-cli

In [ ]:
# Optional module selection for direct (non-air-gapped) execution.
# Supported values:
#   None or "all"                         -> all enabled modules
#   "admin-monitoring"                    -> one module
#   "admin-monitoring,cost-analysis"      -> multiple modules
#   ["admin-monitoring", "cost-analysis"] -> multiple modules
modules_to_deploy = None


# Optional: Name of a Lakehouse in the CURRENT workspace that already contains
# the FOS deployment files under Files/deployment_files/{src,config,data}.
#
# - Leave as None for the normal end-customer flow (files are downloaded
#   from GitHub - no change in behaviour).
# - Set to a Lakehouse name (e.g. via a Fabric Job Run parameter from a
#   GitHub Action) to deploy from files already staged in the workspace,
#   for offline / air-gapped / internal-mirror scenarios.
local_deployment_lakehouse = None
current_branch = None
git_repository_full_name = None



### Import of needed libaries

In [ ]:
import subprocess
import os
import json
from zipfile import ZipFile
import shutil
import re
import requests
import zipfile
from io import BytesIO
import yaml
import sempy.fabric as fabric
import tempfile
import sys
import time
from pathlib import Path

In [ ]:
def configure_module_selection(value):
    if value is None:
        serialized = ""
    elif isinstance(value, str):
        serialized = value.strip()
    else:
        serialized = ",".join(str(module).strip() for module in value if str(module).strip())

    if serialized.lower() == "all":
        serialized = ""
    if local_deployment_lakehouse and serialized:
        raise ValueError(
            "modules_to_deploy is only supported for direct downloads; "
            "air-gapped archives are selected before the notebook runs"
        )

    if serialized:
        os.environ["FOS_MODULE_IDS"] = serialized
        print(f"Requested modules: {serialized}")
    else:
        os.environ.pop("FOS_MODULE_IDS", None)
        print("Requested modules: all enabled modules")

    # Force direct reruns to load the composer from the newly downloaded release.
    sys.modules.pop("compose_release", None)


configure_module_selection(modules_to_deploy)

## Download of source & config files
This part downloads all source and config files of FOS needed for the deployment into the ressources of the notebook

In [ ]:
def download_folder_as_zip(repo_owner, repo_name, output_zip, branch="main", folder_to_extract="src",  remove_folder_prefix = ""):
    # Construct the URL for the GitHub API to download the repository as a zip file
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball/{branch}"

    # Make a request to the GitHub API
    response = requests.get(url)
    response.raise_for_status()

    # Ensure the directory for the output zip file exists
    os.makedirs(os.path.dirname(output_zip), exist_ok=True)

    # Create a zip file in memory
    with zipfile.ZipFile(BytesIO(response.content)) as zipf:
        with zipfile.ZipFile(output_zip, 'w') as output_zipf:
            for file_info in zipf.infolist():
                parts = file_info.filename.split('/')
                if  re.sub(r'^.*?/', '/', file_info.filename).startswith(folder_to_extract):
                    # Extract only the specified folder
                    file_data = zipf.read(file_info.filename)
                    output_zipf.writestr(('/'.join(parts[1:]).replace(remove_folder_prefix, "")), file_data)

def uncompress_zip_to_folder(zip_path, extract_to):
    # Ensure the directory for extraction exists
    os.makedirs(extract_to, exist_ok=True)

    # Uncompress all files from the zip into the specified folder
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    # Delete the original zip file
    os.remove(zip_path)

def download_and_compose_release(repo_owner, repo_name, branch="main"):
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball/{branch}"
    response = requests.get(url)
    response.raise_for_status()

    with tempfile.TemporaryDirectory() as temp_dir:
        checkout_dir = Path(temp_dir) / "checkout"
        checkout_dir.mkdir()
        with zipfile.ZipFile(BytesIO(response.content)) as archive:
            archive.extractall(checkout_dir)
        repo_roots = [path for path in checkout_dir.iterdir() if path.is_dir()]
        if len(repo_roots) != 1:
            raise ValueError("Release archive must contain exactly one repository root")
        repo_root = repo_roots[0]
        sys.path.insert(0, str(repo_root / "cicd" / "release"))
        try:
            from compose_release import compose_release
            composed_dir = Path(temp_dir) / "composed"
            summary = compose_release(repo_root, composed_dir)
        finally:
            sys.path.pop(0)

        for sub in ("config", "data", "src"):
            target = Path("./builtin") / sub
            if target.exists():
                shutil.rmtree(target)
        shutil.copytree(composed_dir / "config", Path("./builtin/config"))
        shutil.copytree(composed_dir / "data", Path("./builtin/data"))
        Path("./builtin/src").mkdir(parents=True)
        with zipfile.ZipFile("./builtin/src/src.zip", "w", zipfile.ZIP_DEFLATED) as archive:
            for source_file in (composed_dir / "src").rglob("*"):
                if source_file.is_file():
                    relative_path = source_file.relative_to(composed_dir / "src")
                    archive.write(source_file, f"src/{relative_path.as_posix()}")
        print(f"Composed modules: {', '.join(summary['modules'])}")


def _resolve_lakehouse_id(workspace_id, lakehouse_name):
    """Resolve a Lakehouse display name to its GUID.

    notebookutils.fs no longer accepts friendly names ('<name>.Lakehouse') in
    ABFSS paths - it fails with 'FriendlyNameSupportDisabled'. We therefore
    look up the Lakehouse ID and use it directly in OneLake URIs.
    """
    for lh in notebookutils.lakehouse.list(workspaceId=workspace_id):
        # Fabric returns dict-like objects; the display name key is 'displayName'
        # and the GUID is 'id'.
        name = lh.get("displayName") if isinstance(lh, dict) else getattr(lh, "displayName", None)
        lh_id = lh.get("id") if isinstance(lh, dict) else getattr(lh, "id", None)
        if name == lakehouse_name:
            return lh_id
    raise ValueError(
        f"Lakehouse '{lakehouse_name}' not found in workspace '{workspace_id}'"
    )

def copy_deployment_files_from_lakehouse(lakehouse_name):
    """Copy src.zip, config.zip and data.zip from a Lakehouse in the current
    workspace into ./builtin/, mirroring the layout produced by the GitHub
    download.

    Used for offline / air-gapped deployments (e.g. GitHub Actions running
    against an internal repo mirror): the CI pipeline uploads three archives
    to a Lakehouse in the target workspace and passes its name via the
    `local_deployment_lakehouse` parameter.

    Expected Lakehouse layout:
        Files/deployment_files/
          - src.zip     (entries prefixed with src/)
          - config.zip  (entries prefixed with config/)
          - data.zip    (entries prefixed with data/)
    """
    ws_id = fabric.get_notebook_workspace_id()
    lakehouse_id = _resolve_lakehouse_id(ws_id, lakehouse_name)
    # Use GUIDs on both sides - friendly names in ABFSS paths are rejected
    # by notebookutils.fs (FriendlyNameSupportDisabled).
    base = (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/"
            f"{lakehouse_id}/Files/deployment_files")

    # Validate presence of required archives (raises if missing)
    required = {"src.zip", "config.zip", "data.zip"}
    present = {e.name for e in notebookutils.fs.ls(base)}
    missing = required - present
    if missing:
        raise FileNotFoundError(
            f"Lakehouse '{lakehouse_name}' is missing deployment archives: {sorted(missing)}"
        )

    # Clean previous state to avoid mixing versions
    for sub in ("config", "data", "src"):
        target = f"./builtin/{sub}"
        if os.path.exists(target):
            shutil.rmtree(target)
    os.makedirs("./builtin/src", exist_ok=True)

    local_root = os.path.abspath("./builtin")

    # src.zip is consumed as-is by copy_to_tmp()
    notebookutils.fs.cp(f"{base}/src.zip", f"file:{local_root}/src/src.zip")

    # config.zip and data.zip are extracted into ./builtin/
    for name in ("config.zip", "data.zip"):
        local_zip = f"{local_root}/{name}"
        notebookutils.fs.cp(f"{base}/{name}", f"file:{local_zip}")
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall(local_root)
        os.remove(local_zip)


if local_deployment_lakehouse:
    # Offline / CI path: reuse deployment files already staged in a Lakehouse
    print(f"Using local deployment files from Lakehouse '{local_deployment_lakehouse}'")
    copy_deployment_files_from_lakehouse(local_deployment_lakehouse)
else:
    # Default end-customer path: download and compose the release repository
    print("Downloading and composing release files from GitHub")
    repo_owner = "GT-Analytics"
    repo_name = "fos_release"
    branch = "main"
    download_and_compose_release(repo_owner, repo_name, branch)


In [ ]:
base_path = './builtin/'
config_path = os.path.join(base_path, 'config/deployment_config.yaml')

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

deploy_order_path = os.path.join(base_path, 'config/deployment_order.json')
with open(deploy_order_path, 'r') as file:
    deployment_order = json.load(file)

src_workspace_name = config['workspace']
source_connections = config.get('connections', {})
connection_specs = config.get('connection_specs', {})
semantic_model_lakehouses = config.get('semantic_model_lakehouses', {})
lakehouse_schemas = config.get('lakehouse_schemas', {})
post_deployment = config.get('post_deployment', {})

mapping_table = []

## Definition of deployment functions

In [ ]:
# Set environment parameters for Fabric CLI
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token

def run_fab_command(command, capture_output: bool = False, silently_continue: bool = False, timeout: int = 300):
    """Run fabric CLI command with timeout protection (default 5 minutes)."""
    try:
        result = subprocess.run(
            ["fab", "-c", command],
            capture_output=capture_output,
            text=True,
            timeout=timeout
        )
        if (not(silently_continue) and (result.returncode > 0 or result.stderr)):
            raise Exception(f"Error running fab command. exit_code: '{result.returncode}'; stderr: '{result.stderr}'")
        if (capture_output):
            output = result.stdout.strip()
            return output
    except subprocess.TimeoutExpired:
        raise Exception(f"Command timed out after {timeout} seconds: {command}")

def fab_get_id(name):
    id = run_fab_command(f"get /{trg_workspace_name}.Workspace/{name} -q id" , capture_output = True, silently_continue= True)
    return(id)

def get_id_by_name(name):
    for it in deployment_order:
        if it.get("name") == name:
                return it.get("fos_id")
    return None


def copy_to_tmp(name):
    """Extract files from zip to memory (handles nested folders at any depth)."""
    path2zip = "./builtin/src/src.zip"
    file_contents = {}  # Store file paths and their content in memory

    with ZipFile(path2zip) as archive:
        for file in archive.namelist():
            # Skip directory entries (ending with /) but include all files at any nesting level
            # This handles: src/name/file.txt, src/name/subfolder/file.txt, src/name/a/b/c/file.txt, etc.
            if file.startswith(f'src/{name}/') and not file.endswith('/'):
                # Read file content into memory instead of extracting to disk
                file_contents[file] = archive.read(file)

    return file_contents

def replace_githubRef_in_memory(file_contents, repo_full_name, branch):
    """Replace GitHub raw content URLs in memory-stored files."""
    updated_contents = {}
    for file_path, content_bytes in file_contents.items():
        file_name = os.path.basename(file_path)
        try:
            content = content_bytes.decode('utf-8')
        except:
            updated_contents[file_path] = content_bytes
            continue

        if file_name.endswith(('.py', '.json', '.ipynb')) and not file_name.endswith('report.json'):
            matches = re.findall(r'https://raw\.githubusercontent\.com/([^/]+/[^/]+/[^/]+/[^/]+/[^/]+)/(.*)', content, re.MULTILINE)
            if matches:
                for match in matches:
                    oldUrl = f"https://raw.githubusercontent.com/{match[0]}/{match[1]}"
                    newUrl = f"https://raw.githubusercontent.com/{repo_full_name}/refs/heads/{branch}/{match[1]}"
                    content = content.replace(oldUrl, newUrl)
        updated_contents[file_path] = content.encode('utf-8')

    return updated_contents

def replace_ids_in_memory(file_contents, mapping_table):
    """Replace IDs in memory-stored files."""
    updated_contents = {}

    for file_path, content_bytes in file_contents.items():
        file_name = os.path.basename(file_path)

        # Decode bytes to string
        try:
            content = content_bytes.decode('utf-8')
        except:
            # If decoding fails, keep as binary
            updated_contents[file_path] = content_bytes
            continue

        if file_name.endswith('.ipynb'):
            notebook_json = json.loads(content)
            dependencies = notebook_json.get('metadata', {}).get('dependencies', {})
            depend = json.dumps(dependencies)
            for mapping in mapping_table:
                depend = depend.replace(mapping["old_id"], mapping["new_id"])
            notebook_json['metadata']['dependencies'] = json.loads(depend)
            content = json.dumps(notebook_json)

        elif file_name.endswith(('.py', '.json', '.pbir', '.platform', '.tmdl')) and not file_name.endswith('report.json'):
            for mapping in mapping_table:
                content = content.replace(mapping["old_id"], mapping["new_id"])

        updated_contents[file_path] = content.encode('utf-8')

    return updated_contents

def write_memory_to_temp(file_contents, temp_dir):
    """Write in-memory files to temporary directory (system temp, not builtin storage)."""
    for file_path, content_bytes in file_contents.items():
        full_path = os.path.join(temp_dir, file_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, 'wb') as f:
            f.write(content_bytes)
    return temp_dir

def get_semantic_model_id_from_memory(file_contents, name):
    """Get semantic model ID from in-memory report definition."""
    definition_path = f'src/{name}/definition.pbir'
    if definition_path in file_contents:
        content = json.loads(file_contents[definition_path].decode('utf-8'))
        semantic_model_id = content.get('datasetReference', {}).get('byConnection', {}).get('pbiModelDatabaseName')
        if semantic_model_id:
            return semantic_model_id
    return None

def get_semantic_model_id(report_folder):
    definition_file = os.path.join(report_folder, 'definition.pbir')
    if os.path.exists(definition_file):
        with open(definition_file, 'r', encoding='utf-8') as file:
            content = json.load(file)
            semantic_model_id = content.get('datasetReference', {}).get('byConnection', {}).get('pbiModelDatabaseName')
            if semantic_model_id:
                return semantic_model_id
    return None

def update_sm_connection_to_fos_lakehouse_in_memory(file_contents, name):
    """Update semantic model connection to FOS lakehouse in memory."""
    new_sm_db = run_fab_command(f"get /{trg_workspace_name}.Workspace/FOS_Lakehouse.Lakehouse -q properties.sqlEndpointProperties.connectionString", capture_output=True, silently_continue=True)
    new_lakehouse_sql_id = run_fab_command(f"get /{trg_workspace_name}.Workspace/FOS_Lakehouse.Lakehouse -q properties.sqlEndpointProperties.id", capture_output=True, silently_continue=True)

    expressions_path = f'src/{name}/definition/expressions.tmdl'
    if expressions_path in file_contents:
        content = file_contents[expressions_path].decode('utf-8')
        match = re.search(r'Sql\.Database\("([^"]+)",\s*"([^"]+)"\)', content)
        if match:
            old_sm_db, old_lakehouse_sql_id = match.group(1), match.group(2)
            content = content.replace(old_sm_db, new_sm_db).replace(old_lakehouse_sql_id, new_lakehouse_sql_id)
            file_contents[expressions_path] = content.encode('utf-8')
    return file_contents

def update_sm_connection_to_fos_lakehouse(semantic_model_folder):
    new_sm_db= run_fab_command(f"get /{trg_workspace_name}.Workspace/FOS_Lakehouse.Lakehouse -q properties.sqlEndpointProperties.connectionString", capture_output = True, silently_continue=True)
    new_lakehouse_sql_id= run_fab_command(f"get /{trg_workspace_name}.Workspace/FOS_Lakehouse.Lakehouse -q properties.sqlEndpointProperties.id", capture_output = True, silently_continue=True)

    expressions_file = os.path.join(semantic_model_folder, 'definition', 'expressions.tmdl')
    if os.path.exists(expressions_file):
        with open(expressions_file, 'r', encoding='utf-8') as file:
            content = file.read()
            match = re.search(r'Sql\.Database\("([^"]+)",\s*"([^"]+)"\)', content)
            if match:
                old_sm_db, old_lakehouse_sql_id = match.group(1), match.group(2)
                content = content.replace(old_sm_db, new_sm_db).replace(old_lakehouse_sql_id, new_lakehouse_sql_id)
                with open(expressions_file, 'w', encoding='utf-8') as file:
                    file.write(content)


def update_report_definition_in_memory(file_contents, name):
    """Update report definition in memory to reference semantic model by ID."""
    semantic_model_id = get_semantic_model_id_from_memory(file_contents, name)
    definition_path = f"src/{name}/definition.pbir"

    if definition_path in file_contents and semantic_model_id:
        report_definition = json.loads(file_contents[definition_path].decode('utf-8'))

        # Update connection string to reference the semantic model by ID
        # Format: Data Source=powerbi://api.powerbi.com/v1.0/myorg/{workspace};initial catalog={model};semanticmodelid={id}
        connection_string = f"Data Source=powerbi://api.powerbi.com/v1.0/myorg/{trg_workspace_name};initial catalog={{MODEL_NAME}};integrated security=ClaimsToken;semanticmodelid={semantic_model_id}"

        # Ensure datasetReference structure exists
        if "datasetReference" not in report_definition:
            report_definition["datasetReference"] = {}

        # Clear byPath if it exists
        if "byPath" in report_definition["datasetReference"]:
            del report_definition["datasetReference"]["byPath"]

        # Set byConnection with only the connectionString property
        report_definition["datasetReference"]["byConnection"] = {
            "connectionString": connection_string
        }

        file_contents[definition_path] = json.dumps(report_definition, indent=4).encode('utf-8')

    return file_contents

def update_report_definition(path):
    """Update report definition to reference semantic model by ID (legacy disk-based version)."""
    semantic_model_id = get_semantic_model_id(path)
    definition_path = os.path.join(path, "definition.pbir")

    if semantic_model_id:
        with open(definition_path, "r", encoding="utf8") as file:
            report_definition = json.load(file)

        # Update connection string to reference the semantic model by ID
        connection_string = f"Data Source=powerbi://api.powerbi.com/v1.0/myorg/{trg_workspace_name};initial catalog={{MODEL_NAME}};integrated security=ClaimsToken;semanticmodelid={semantic_model_id}"

        # Ensure datasetReference structure exists
        if "datasetReference" not in report_definition:
            report_definition["datasetReference"] = {}

        # Clear byPath if it exists
        if "byPath" in report_definition["datasetReference"]:
            del report_definition["datasetReference"]["byPath"]

        # Set byConnection with only the connectionString property
        report_definition["datasetReference"]["byConnection"] = {
            "connectionString": connection_string
        }

        with open(definition_path, "w") as file:
            json.dump(report_definition, file, indent=4)

def print_color(text, state):
    red  = '\033[91m'
    yellow = '\033[93m'
    green = '\033[92m'
    white = '\033[0m'
    if state == "error":
        print(red, text, white)
    elif state == "warning":
        print(yellow, text, white)
    elif state == "success":
        print(green, text, white)
    else:
        print("", text)



In [ ]:
def update_sm_connection_to_lakehouse_in_memory(file_contents, name, lakehouse_name):
    """Point a semantic model at its module-declared Lakehouse SQL endpoint."""
    new_sm_db = run_fab_command(
        f"get /{trg_workspace_name}.Workspace/{lakehouse_name} -q properties.sqlEndpointProperties.connectionString",
        capture_output=True,
        silently_continue=True,
    )
    new_lakehouse_sql_id = run_fab_command(
        f"get /{trg_workspace_name}.Workspace/{lakehouse_name} -q properties.sqlEndpointProperties.id",
        capture_output=True,
        silently_continue=True,
    )

    expressions_path = f'src/{name}/definition/expressions.tmdl'
    if expressions_path in file_contents:
        content = file_contents[expressions_path].decode('utf-8')
        match = re.search(r'Sql\.Database\("([^"]+)",\s*"([^"]+)"\)', content)
        if match:
            old_sm_db, old_lakehouse_sql_id = match.group(1), match.group(2)
            content = content.replace(old_sm_db, new_sm_db).replace(
                old_lakehouse_sql_id, new_lakehouse_sql_id
            )
            file_contents[expressions_path] = content.encode('utf-8')
    return file_contents

## Creation of connections

In [ ]:
def create_or_get_connection(name, base_url, audience):
    try:
        run_fab_command(
            f"""create .connections/{name}.connection
            -P connectionDetails.type=WebForPipeline,connectionDetails.creationMethod=WebForPipeline.Contents,connectionDetails.parameters.baseUrl={base_url},connectionDetails.parameters.audience={audience},credentialDetails.type=Anonymous"""
        )
        print_color("New connection created. Enter service principal credentials", "success")
    except Exception:
        print_color("Connection already exists", "warning")

    connection_id = run_fab_command(
        f"get .connections/{name}.Connection -q id",
        silently_continue=True,
        capture_output=True,
    )
    print(f"Connection ID ({name}): {connection_id}")
    return connection_id


for connection_key, connection_name in source_connections.items():
    spec = connection_specs.get(connection_key)
    if spec is None:
        raise ValueError(f"Missing connection_specs entry for '{connection_key}'")
    connection_id = create_or_get_connection(
        connection_name,
        spec['base_url'],
        spec['audience'],
    )
    mapping_table.append({
        "old_id": get_id_by_name(connection_name),
        "new_id": connection_id,
    })

## Get current Workspace
This cell gets the current workspace to deploy FOS automatically inside it

In [ ]:
trg_workspace_id = fabric.get_notebook_workspace_id()
res = run_fab_command(f"api -X get workspaces/{trg_workspace_id}" , capture_output = True, silently_continue=True)
trg_workspace_name = json.loads(res)["text"]["displayName"]

print(f"Current workspace: {trg_workspace_name}")
print(f"Current workspace ID: {trg_workspace_id}")


mapping_table.append({ "old_id": get_id_by_name(src_workspace_name + ".Workspace"), "new_id": trg_workspace_id })
mapping_table.append({ "old_id": "00000000-0000-0000-0000-000000000000", "new_id": trg_workspace_id })

In [ ]:
mapping_table

## Deployment Logic
This part iterates through all the items, gets the respective source code, replaces all IDs dynamically and deploys the new item

In [ ]:
exclude = [src_workspace_name + ".Workspace", *source_connections.values()]

for it in deployment_order:
    new_id = None
    name = it["name"]

    if name in exclude:
        continue

    print("")
    print("#############################################")
    print(f"Deploying {name}")

    file_contents = copy_to_tmp(name)
    file_contents = replace_ids_in_memory(file_contents, mapping_table)
    if git_repository_full_name and current_branch:
        file_contents = replace_githubRef_in_memory(file_contents, git_repository_full_name, current_branch)

    cli_parameter = ''
    if "Notebook" in name:
        cli_parameter += " --format .ipynb"
    elif "Lakehouse" in name:
        run_fab_command(
            f"create /{trg_workspace_name}.Workspace/{name} -P enableSchemas=True",
            silently_continue=True,
        )
        new_id = fab_get_id(name)
        default_schema = run_fab_command(
            f"get /{trg_workspace_name}.Workspace/{name} -q properties.defaultSchema",
            capture_output=True,
            silently_continue=True,
        )
        if (default_schema or '').strip().lower() != 'dbo':
            raise RuntimeError(
                f"Lakehouse '{name}' is not schema-enabled. Existing non-schema "
                "Lakehouses cannot currently be converted in place; create a new "
                "schema-enabled Lakehouse and migrate its data before deployment."
            )
        declared_schemas = lakehouse_schemas.get(name, [])
        if declared_schemas:
            print(
                "Schemas will be created by post-deployment initialization jobs: "
                + ", ".join(declared_schemas)
            )
        mapping_table.append({"old_id": get_id_by_name(name), "new_id": new_id})
        continue
    elif "Report" in name:
        file_contents = update_report_definition_in_memory(file_contents, name)
    elif name in semantic_model_lakehouses:
        file_contents = update_sm_connection_to_lakehouse_in_memory(
            file_contents,
            name,
            semantic_model_lakehouses[name],
        )

    with tempfile.TemporaryDirectory() as temp_dir:
        try:
            write_memory_to_temp(file_contents, temp_dir)
            item_path = os.path.join(temp_dir, f"src/{name}")

            print(f"Importing {name}...")
            run_fab_command(
                f"import /{trg_workspace_name}.Workspace/{name} -i {item_path} -f {cli_parameter}",
                silently_continue=True,
                timeout=600,
            )
            new_id = fab_get_id(name)
            mapping_table.append({"old_id": it["fos_id"], "new_id": new_id})
            print_color(f"Successfully deployed {name}", "success")
        except subprocess.TimeoutExpired as ex:
            print_color(f"Timeout importing {name}: {ex}", "error")
            raise
        except Exception as ex:
            print_color(f"Error deploying {name}: {ex}", "error")
            raise

## Move items into folders
The items will be moved into the respective folders. Definition is done in the deployment_config.yml

In [ ]:
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token

def api_payload(command):
    response = json.loads(run_fab_command(command, capture_output=True))
    payload = response.get('text', response)
    return json.loads(payload) if isinstance(payload, str) else payload

items_in_ws = api_payload(f'api /workspaces/{trg_workspace_id}/items')['value']
item_ids = {item['displayName'] + '.' + item['type']: item['id'] for item in items_in_ws}
existing_folders = api_payload(f'api -X get workspaces/{trg_workspace_id}/folders?recursive=true').get('value', [])
folder_ids = {(folder.get('parentFolderId'), folder['displayName']): folder['id'] for folder in existing_folders}

def ensure_folder(folder_name, parent_id=None):
    key = (parent_id, folder_name)
    if key in folder_ids:
        return folder_ids[key]
    body = {'displayName': folder_name}
    if parent_id:
        body['parentFolderId'] = parent_id
    folder = api_payload(f"api -X post workspaces/{trg_workspace_id}/folders -i '{json.dumps(body)}'")
    folder_ids[key] = folder['id']
    return folder['id']

def deploy_folders(folders, parent_id=None, path=()):
    for folder in folders:
        folder_path = path + (folder['name'],)
        folder_id = ensure_folder(folder['name'], parent_id)
        found_ids = [item_ids[name] for name in folder.get('items', []) if name in item_ids]
        for name in (name for name in folder.get('items', []) if name not in item_ids):
            print_color(f"Item not found for folder {' / '.join(folder_path)}: {name}", 'warning')
        if found_ids:
            body = {'targetFolderId': folder_id, 'items': found_ids}
            api_payload(f"api -X post workspaces/{trg_workspace_id}/items/bulkmove -i '{json.dumps(body)}'")
        deploy_folders(folder.get('folders', []), folder_id, folder_path)

deploy_folders(config.get('folders', []))

## Post-Deployment logic
In this separate notebook, all needed tables for FOS are automatically deployed. Addtionally new columns will be added to lakehouse tables in order to be available for the semantic model. This notebook has been deployed from the source code in the step before

In [ ]:
# Set environment parameters for Fabric CLI
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token


In [ ]:
table_definitions_path = "./builtin/data/table_definitions.snappy.parquet"


In [ ]:
fos_lakehouse_id = fab_get_id('FOS_Lakehouse.Lakehouse')
if fos_lakehouse_id and os.path.exists(table_definitions_path):
    default_fs = notebookutils.conf.get("fs.defaultFS")
    onelake_endpoint = default_fs.split("@", 1)[1].rstrip("/")
    target_directory = (
        f"abfss://{trg_workspace_id}@{onelake_endpoint}/"
        f"{fos_lakehouse_id}/Files/FOS/config"
    )
    target_path = f"{target_directory}/table_definitions.snappy.parquet"
    notebookutils.fs.mkdirs(target_directory)
    try:
        notebookutils.fs.rm(target_path)
    except Exception:
        pass
    copied = notebookutils.fs.cp(
        f"file:{os.path.abspath(table_definitions_path)}",
        target_path,
    )
    if copied is False:
        raise RuntimeError(f"Failed to stage table definitions at {target_path}")
    print("Staged Admin Monitoring table definitions in FOS_Lakehouse Files")

In case the last step fails, please try to run it again or go to the Init_FOS_Lakehouse_Tables Notebook and run it manually

In [ ]:
items_response = run_fab_command(
    f'api -X get -A fabric /workspaces/{trg_workspace_id}/items',
    capture_output=True,
)
workspace_items = json.loads(items_response)['text']['value']
sql_endpoint_ids = {}

for lakehouse_name in post_deployment.get('sql_endpoints', []):
    display_name = lakehouse_name.removesuffix('.Lakehouse')
    endpoint = next(
        (
            item for item in workspace_items
            if item['displayName'] == display_name and item['type'] == 'SQLEndpoint'
        ),
        None,
    )
    if endpoint is None:
        print_color(f"SQL endpoint not found for {lakehouse_name}", "warning")
        continue
    sql_endpoint_ids[lakehouse_name] = endpoint['id']
    print(f"{lakehouse_name} SQL Endpoint ID: {endpoint['id']}")

In [ ]:
time.sleep(10)
for job in post_deployment.get('jobs', []):
    item_name = job['item']
    command = f"job run {trg_workspace_name}.Workspace/{item_name}"
    parameters = job.get('parameters')
    if parameters:
        command += " -i " + json.dumps({"parameters": parameters})
    print(f"Running post-deployment job: {item_name}")
    run_fab_command(command, timeout=3600)

In [ ]:
for lakehouse_name, endpoint_id in sql_endpoint_ids.items():
    try:
        run_fab_command(
            f'api -A fabric -X post workspaces/{trg_workspace_id}/sqlEndpoints/{endpoint_id}/refreshMetadata?preview=True -i {{}}',
            capture_output=True,
        )
        print(f"Refreshed SQL endpoint: {lakehouse_name}")
    except Exception:
        print_color(
            f"SQL endpoint refresh failed for {lakehouse_name}; the API is still in preview",
            "warning",
        )

for semantic_model in semantic_model_lakehouses:
    semantic_model_id = run_fab_command(
        f"get /{trg_workspace_name}.Workspace/{semantic_model} -q id",
        capture_output=True,
        silently_continue=True,
    )
    run_fab_command(
        f'api -A powerbi -X post datasets/{semantic_model_id}/refreshes -i {{"retryCount":"3"}}'
    )